# Notebook A — Convert the whole NIH ChestX-ray14 dataset into numbers and store it on Google Drive

**What this notebook does (run it once)**
1. Downloads NIH ChestX-ray14 (about 42 GB) and reads the labels / metadata and the **official train-test split**.
2. Resizes **every image to 224 x 224** (CLAHE contrast normalisation) and saves them as compact `uint8` number arrays on Drive (optional).
3. Passes every image once through a pretrained **DenseNet-121** and saves one **1024-number embedding per image** (float16). This is the "compressed" form of the data.
4. Saves `meta.csv`, `emb_all.npy`, bounding boxes and a `manifest.json` that **Notebook B** loads for training.

**It is resumable.** Work is saved in shards of 4,096 images; if Colab disconnects, re-run the cells and finished shards are skipped.

**Before you run**
- Use a **GPU runtime** (T4 is fine) and check free disk with the first cell. The raw download needs about 42 GB or more of local disk during extraction.
- Drive space needed: about 5.6 GB for the image arrays (`SAVE_IMAGES = True`) plus about 0.25 GB for embeddings. Set `SAVE_IMAGES = False` if your Drive is nearly full (then Notebook B can only train on embeddings, not fine-tune).
- First time, set `DATA_SOURCE = "sample"` (5.6k images, a few minutes) to confirm everything works, then switch to `"full"`.

In [2]:
# Install CLI tools
!pip install -q kaggle kagglehub opencv-python-headless
!df -h /content | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


overlay         113G   43G   71G  38% /


## 0. Configuration

In [1]:
import os, time, math, json, shutil, hashlib, warnings, subprocess
import numpy as np, pandas as pd
import torch, torch.nn as nn, torchvision
warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION (Tuned for A100 40GB / High-RAM / Drive limits)
# ==============================================================================
# DATA_SOURCE options:
#   'sample' : 5,606 images (~2 GB). Runs in 2-3 minutes! No kaggle.json needed.
#   'full'   : 112,120 images (42 GB). Official NIH dataset. Requires kaggle.json.
DATA_SOURCE = 'full'

# Resolution: 512x512 is the sweet spot for your 40GB A100 GPU!
RES         = 512
SHARD       = 4096        # Images per saved shard

# GOOGLE DRIVE STORAGE CONTROL:
# Free Google Drive has 15 GB limit.
#   SAVE_IMAGES = False -> Saves ONLY the 1024-d DenseNet embeddings (~0.23 GB total!). Fits in ANY Drive.
#   SAVE_IMAGES = True  -> Saves both 512x512 images (~29.4 GB) and embeddings. (Requires 30GB+ Drive).
SAVE_IMAGES = False

# LOCAL COLAB DISK CONTROL:
#   DELETE_RAW = True deletes unzipped raw PNG images after embeddings are saved, freeing all local disk.
DELETE_RAW  = True
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/hybrid_pgm_data' + ('_sample' if DATA_SOURCE == 'sample' else '')
os.makedirs(WORK, exist_ok=True)

# Hardware Detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# Adaptive Batch Size based on GPU VRAM:
BATCH_SIZE = 64 if vram_gb >= 35 else (32 if vram_gb >= 14 else 16)

print(f'GPU Device: {device} ({vram_gb:.1f} GB VRAM) | Batch Size: {BATCH_SIZE}')
print(f'Dataset Mode: {DATA_SOURCE} | Resolution: {RES}x{RES}')
print(f"Google Drive Folder: {WORK} (SAVE_IMAGES={SAVE_IMAGES}, Space: ~{'29.4 GB' if SAVE_IMAGES else '0.23 GB'})")


Mounted at /content/drive
Device: cuda | source: full | Drive folder: /content/drive/MyDrive/hybrid_pgm_data


## 1. Download and build the metadata table

In [ ]:
# ==============================================================================
# 1. DOWNLOAD & BUILD METADATA (SEQUENTIAL CHUNKING - NO DISK CRASH)
# ==============================================================================
import zipfile

root = '/content/nih_raw_data'
images_dir = os.path.join(root, 'images')
os.makedirs(images_dir, exist_ok=True)

pngs = {}

if DATA_SOURCE == 'sample':
    print('Downloading NIH sample dataset (5.6k images) via kagglehub...')
    import kagglehub
    slug = 'nih-chest-xrays/sample'
    sample_root = kagglehub.dataset_download(slug)
    csvs, txts = {}, {}
    for r, _, fs in os.walk(sample_root):
        for f in fs:
            p = os.path.join(r, f); fl = f.lower()
            if fl.endswith('.png'): pngs[f] = p
            elif fl.endswith('.csv'): csvs[f] = p
            elif fl.endswith('.txt'): txts[f] = p
    meta_csv_path = csvs.get('sample_labels.csv', os.path.join(sample_root, 'sample_labels.csv'))
    print(f'Sample images successfully downloaded and indexed: {len(pngs)}')

else:
    # FULL NIH DATASET (112,120 IMAGES / 12 ZIPS)
    # Step A: Kaggle API Credentials Setup
    kaggle_json_path = '/content/kaggle.json'
    if not os.path.exists(kaggle_json_path) and not os.path.exists('/root/.kaggle/kaggle.json'):
        print('=' * 70)
        print('⚠️ KAGGLE API KEY REQUIRED FOR FULL 112K DATASET:')
        print('1. Go to https://www.kaggle.com/settings -> Click "Create New Token"')
        print('2. Upload the downloaded "kaggle.json" file to the /content/ folder in Colab.')
        print('3. Make sure you have accepted the dataset rules at:')
        print('   https://www.kaggle.com/datasets/nih-chest-xrays/data')
        print('=' * 70)
        raise FileNotFoundError('Please upload kaggle.json to /content/ before running!')

    if os.path.exists(kaggle_json_path):
        os.makedirs('/root/.kaggle', exist_ok=True)
        shutil.copy(kaggle_json_path, '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
        print('✓ Kaggle API credentials authenticated successfully.')

    # Step B: Download metadata CSVs and official split list
    print('\nDownloading dataset metadata and split files...')
    for m_file in ['Data_Entry_2017.csv', 'test_list.txt', 'BBox_List_2017.csv']:
        cmd = ['kaggle', 'datasets', 'download', 'nih-chest-xrays/data', '--file', m_file, '-p', root, '--unzip']
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.returncode != 0:
            print(f'Warning downloading {m_file}: {res.stderr.strip() or res.stdout.strip()}')
        else:
            print(f'  ✓ Downloaded {m_file}')

    meta_csv_path = os.path.join(root, 'Data_Entry_2017.csv')
    if not os.path.exists(meta_csv_path):
        # Check if downloaded as zip without auto-unzip
        for f in os.listdir(root):
            if f.endswith('.zip'):
                with zipfile.ZipFile(os.path.join(root, f), 'r') as zr: zr.extractall(root)
                os.remove(os.path.join(root, f))

    assert os.path.exists(meta_csv_path), f'Failed to download {meta_csv_path}! Check Kaggle API authentication.'

    # Step C: Sequential 1-by-1 Download of Image Archives (Prevents Colab Disk Crash!)
    print('\n' + '=' * 70)
    print('STARTING SEQUENTIAL DOWNLOAD (1 archive at a time to stay under 48GB disk)...')
    print('=' * 70)

    for i in range(1, 13):
        zip_name = f'images_{i:03d}.zip'
        print(f'\n[Archive {i}/12] Downloading {zip_name}...')
        cmd = ['kaggle', 'datasets', 'download', 'nih-chest-xrays/data', '--file', zip_name, '-p', root]
        res = subprocess.run(cmd, capture_output=True, text=True)
        
        if res.returncode != 0:
            print(f'  ⚠️ Error from Kaggle CLI: {res.stderr.strip() or res.stdout.strip()}')
            # Try alternative command format without --file
            cmd_alt = ['kaggle', 'datasets', 'download', 'nih-chest-xrays/data', '-f', zip_name, '-p', root]
            res_alt = subprocess.run(cmd_alt, capture_output=True, text=True)
            if res_alt.returncode != 0:
                print(f'  ⚠️ Alternative download failed: {res_alt.stderr.strip()}')

        # Look for the downloaded zip (can be named images_001.zip or images_001.zip.zip)
        target_zip = None
        for candidate in [zip_name, zip_name + '.zip']:
            cp = os.path.join(root, candidate)
            if os.path.exists(cp):
                target_zip = cp; break

        if target_zip and os.path.exists(target_zip):
            zip_sz_gb = os.path.getsize(target_zip) / 1e9
            print(f'  Extracting {os.path.basename(target_zip)} ({zip_sz_gb:.2f} GB) to {images_dir}...')
            with zipfile.ZipFile(target_zip, 'r') as zr:
                zr.extractall(images_dir)
            os.remove(target_zip)
            print(f'  ✓ Extracted and deleted {os.path.basename(target_zip)} to reclaim local disk space.')
        else:
            print(f'  ⚠️ Could not find extracted file for {zip_name}.')

    # Index all extracted PNGs
    for r, _, fs in os.walk(images_dir):
        for f in fs:
            if f.lower().endswith('.png'): pngs[f] = os.path.join(r, f)
    print(f'\nTotal extracted images successfully indexed: {len(pngs)}')

# ==============================================================================
# VALIDATION CHECK: Prevent downstream empty-array crash!
# ==============================================================================
assert len(pngs) > 0, (
    f'CRITICAL ERROR: No images found! (len(pngs) = 0)\n'
    'Kaggle download did not extract any PNG files. Please check:\n'
    '1. Is kaggle.json uploaded to /content/kaggle.json?\n'
    '2. Have you accepted dataset rules on https://www.kaggle.com/datasets/nih-chest-xrays/data ?\n'
    '3. Tip: If you want a fast 2-minute test run without downloading 42GB, set DATA_SOURCE = "sample" in Section 0!'
)

# Build and Clean Metadata DataFrame
meta = pd.read_csv(meta_csv_path)
meta.columns = [c.strip() for c in meta.columns]
meta = meta.rename(columns={'Image Index': 'file', 'Finding Labels': 'finding', 'Patient ID': 'patient',
                            'Patient Age': 'age', 'Patient Gender': 'sex', 'View Position': 'view'})
meta = meta[['file', 'finding', 'patient', 'age', 'sex', 'view']]
meta = meta[meta.file.isin(pngs)].sort_values('file').reset_index(drop=True)

assert len(meta) > 0, f'Metadata table has 0 matching files with indexed PNGs! Total PNGs: {len(pngs)}'

# Data Cleaning
meta['age'] = pd.to_numeric(meta.age.astype(str).str.extract(r'(\d+)')[0], errors='coerce')
meta.loc[meta.age > 100, 'age'] = np.nan
meta['age'] = meta.age.fillna(meta.age.median())
meta['is_normal'] = meta.finding.str.strip() == 'No Finding'
meta['fam_alv'] = meta.finding.str.contains('Pneumonia|Infiltration|Consolidation')
meta['fam_eff'] = meta.finding.str.contains('Effusion|Atelectasis')
meta['fam_nod'] = meta.finding.str.contains('Nodule|Mass')

# Map Official Patient-Level Train/Test Split
if DATA_SOURCE == 'full':
    tl_path = os.path.join(root, 'test_list.txt')
    if os.path.exists(tl_path):
        tl = set(open(tl_path).read().split())
        meta['split_official'] = np.where(meta.file.isin(tl), 'test', 'trainval')
    else:
        meta['split_official'] = 'trainval'
else:
    rs = np.random.default_rng(0); pats = rs.permutation(meta.patient.unique())
    test_p = set(pats[:int(0.25 * len(pats))])
    meta['split_official'] = np.where(meta.patient.isin(test_p), 'test', 'trainval')

meta.insert(0, 'idx', np.arange(len(meta)))
meta.to_csv(f'{WORK}/meta.csv', index=False)

bbox_file = os.path.join(root, 'BBox_List_2017.csv')
if os.path.exists(bbox_file):
    shutil.copy(bbox_file, f'{WORK}/bbox_list.csv')
    print('✓ Copied bounding-box reference file to Google Drive.')

print(f'\n✓ Metadata successfully saved to {WORK}/meta.csv ({len(meta)} records)')
print('Splits:', meta.split_official.value_counts().to_dict(), '| Abnormal rate:', round(float((~meta.is_normal).mean()), 3))


100%|██████████| 42.0G/42.0G [07:12<00:00, 104MB/s]

Extracting files...


## 2. Images -> numbers (resized arrays) and embeddings, shard by shard
Every shard is saved to Drive as soon as it is finished (`img_shard_XXXX.npy`, `emb_shard_XXXX.npy`). Re-running the cell skips shards that already exist.

In [ ]:
# ==============================================================================
# 2. IMAGES -> NUMBERS (RESIZED ARRAYS & DENSENET EMBEDDINGS) SHARD BY SHARD
# ==============================================================================
import cv2
from concurrent.futures import ThreadPoolExecutor
bad_reads = []

n = len(meta)
assert n > 0, 'Cannot process an empty metadata table! Please ensure Section 1 downloaded images.'
nsh = math.ceil(n / SHARD)
assert nsh > 0, f'Number of shards must be > 0 (got {nsh})'

def load_one(path):
    a = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if a is None: bad_reads.append(path); return np.zeros((RES, RES), np.uint8)
    a = cv2.resize(a, (RES, RES), interpolation=cv2.INTER_AREA)
    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(a)

# ImageNet Normalization Constants
imn_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
imn_std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

print('Loading DenseNet-121 backbone (channels_last, FP16 AMP)...')
try: bb = torchvision.models.densenet121(weights=torchvision.models.DenseNet121_Weights.IMAGENET1K_V1)
except Exception: bb = torchvision.models.densenet121(pretrained=True)
bb.classifier = nn.Identity()
bb = bb.to(device).to(memory_format=torch.channels_last).eval()

@torch.no_grad()
def extract(U, bs=BATCH_SIZE):
    out = []
    for i in range(0, len(U), bs):
        x = torch.from_numpy(U[i:i+bs]).to(device).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
        x = ((x - imn_mean) / imn_std).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            out.append(bb(x).float().cpu())
    return torch.cat(out).numpy().astype(np.float16)

t_all = time.time()
print(f'Processing {n} images across {nsh} shards (Shard size: {SHARD}, Resolution: {RES}x{RES}, Batch size: {BATCH_SIZE})...\n')

for s in range(nsh):
    ip = f'{WORK}/img_shard_{s:04d}.npy'
    ep = f'{WORK}/emb_shard_{s:04d}.npy'
    need_img = SAVE_IMAGES and not os.path.exists(ip)
    need_emb = not os.path.exists(ep)
    
    # Check if already processed (Resumption support!)
    if not (need_img or need_emb):
        print(f'  Shard {s+1}/{nsh} already exists on Drive. Skipping...')
        continue
        
    sub = meta.iloc[s * SHARD:(s + 1) * SHARD]; t0 = time.time()
    with ThreadPoolExecutor(16) as ex:
        imgs = np.stack(list(ex.map(load_one, [pngs[f] for f in sub.file])))
    
    if need_img:
        np.save(ip + '.tmp.npy', imgs)
        os.replace(ip + '.tmp.npy', ip)
        
    if need_emb:
        np.save(ep + '.tmp.npy', extract(imgs, bs=BATCH_SIZE))
        os.replace(ep + '.tmp.npy', ep)
        
    elapsed = time.time() - t0
    total_elapsed = time.time() - t_all
    print(f'  ✓ Shard {s+1}/{nsh} saved ({len(sub)} images, {elapsed:.1f}s | Total: {total_elapsed/60:.1f} min)')

print(f'\nFeature extraction complete! Unreadable images replaced by blanks: {len(bad_reads)}')


## 3. Consolidate and verify

In [ ]:
# ==============================================================================
# 3. CONSOLIDATE EMBEDDINGS & WRITE MANIFEST (VERIFICATION)
# ==============================================================================
print('Consolidating all shard embeddings into single matrix...')
emb_list = [np.load(f'{WORK}/emb_shard_{s:04d}.npy') for s in range(nsh) if os.path.exists(f'{WORK}/emb_shard_{s:04d}.npy')]

assert len(emb_list) > 0, (
    f'CRITICAL ERROR: No embedding shards found in {WORK}!\n'
    'Cannot concatenate empty list. Please verify Section 2 extracted shards without errors.'
)
assert len(emb_list) == nsh, f'Missing shards! Expected {nsh}, found {len(emb_list)}'

emb = np.concatenate(emb_list)
assert len(emb) == len(meta), f'Length mismatch: {len(emb)} vs {len(meta)}'
assert np.isfinite(emb.astype(np.float32)).all(), 'Non-finite values found in embeddings'

emb_all_path = f'{WORK}/emb_all.npy'
np.save(emb_all_path, emb)
print(f'✓ Saved master embeddings: {emb_all_path} (Shape: {emb.shape}, Size: {os.path.getsize(emb_all_path)/1e6:.2f} MB)')

# Generate Manifest Metadata JSON
manifest = {
    'n': int(len(meta)),
    'res': RES,
    'shard': SHARD,
    'n_shards': int(nsh),
    'backbone': 'densenet121-imagenet1k-v1 (global-average-pooled, 1024-d, FP16)',
    'emb_dim': int(emb.shape[1]),
    'emb_dtype': 'float16',
    'save_images': bool(SAVE_IMAGES),
    'data_source': DATA_SOURCE,
    'clahe': True,
    'unreadable_images': len(bad_reads),
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
}
json.dump(manifest, open(f'{WORK}/manifest.json', 'w'), indent=2)
print('✓ Saved manifest.json:')
print(json.dumps(manifest, indent=2))

# Verify Google Drive Folder Size
drive_files = sorted(os.listdir(WORK))
tot_size_bytes = sum(os.path.getsize(os.path.join(WORK, f)) for f in drive_files if os.path.isfile(os.path.join(WORK, f)))
print(f'\nGoogle Drive Storage Used: {tot_size_bytes/1e9:.2f} GB')
print(f'Stored files: {len(drive_files)} items in {WORK}')

# Clean up raw local images if requested
if DELETE_RAW and os.path.exists(root):
    shutil.rmtree(root, ignore_errors=True)
    print('✓ Deleted raw downloaded images from local Colab disk. Disk space fully reclaimed!')

print('\n🎉 NOTEBOOK A COMPLETED SUCCESSFULLY! READY FOR NOTEBOOK B.')


## Done
Your Drive folder now holds `meta.csv`, `emb_all.npy`, `manifest.json`, optional `bbox_list.csv` and the image/embedding shards. Open **Notebook B** and set its `WORK` path to the same folder.

Notes
- Embeddings come from an **ImageNet-pretrained** DenseNet-121. If you later switch to a chest-X-ray-pretrained backbone, check that it was not trained on NIH images, otherwise the test split is contaminated.
- The embedding is a lossy compression: small findings can be washed out. That is why Notebook B can also fine-tune on the image shards.